## Putting the entire thing together

- Generate questions using LLM
- Pass these questions to the tech assistant LLM
- Gather responses, and evaluate them using the evals asistant LLM
- Print the results


### Imports & Constants

In [158]:
import os
import json
import pickle
import secrets
import random
from dotenv import load_dotenv
from datetime import datetime
from pathlib import Path
from typing import List, Dict, Any, Callable
from pydantic import BaseModel
import pydantic_ai as pda
from sentence_transformers import SentenceTransformer

load_dotenv()

True

In [145]:
LOG_DIR = Path("logs/full_experiment")
# os.makedirs(LOG_DIR, exist_ok=True)

In [146]:
NUM_QUESTIONS = 50

### Define our data models

In [147]:
class QuestionList(BaseModel):
    questions: List[str]

In [148]:
class EvalsCheck(BaseModel):
    check_name: str
    justification: str
    check_pass: bool

In [149]:
class EvalsChecklist(BaseModel):
    checks: List[EvalsCheck]
    verdict: str

### Define system & user prompts

In [150]:
quesgen_system_prompt = """
You are helping to create test questions for an AI agent that answers questions
about basic coding, software development and beginner level Machine Learning.
You are to use the content that's fed to you, which are snippets from the following github repos:
https://www.github.com/freeCodeCamp/freeCodeCamp
https://www.github.com/microsoft/ML-For-Beginners

Generate {n_questions} unique, realistic questions that users who are learning a course based on these
github repos might ask.
The questions should:
- Be natural and varied in style
- Range from simple to complex
- Include both specific technical questions and general course questions
Generate one question for each record.

ALL QUESTIONS MUST BE IN ENGLISH.
NO QUESTIONS MUST BE DUPLICATE.
ALL QUESTIONS MUST BE SPECIFICALLY RELATED TO THE CHUNK OF TEXT FED TO YOU.
NO QUESTIONS MUST BE ON TOPICS THAT ARE NOT COVERED IN THE CHUNK OF TEXT FED TO YOU.
""".strip()

In [151]:
evals_system_prompt = """
Use this checklist to evaluate the quality of an AI agent's answer (<ANSWER>)
to a user question (<QUESTION>).
We also include the entire log (<LOG>) for analysis.
For each item, check if the condition is met.
Checklist:
- instructions_follow: The agent followed the user's instructions (in <INSTRUCTIONS>)
- instructions_avoid: The agent avoided doing things it was told not to do
- answer_relevant: The response directly addresses the user's question
- answer_clear: The answer is clear and correct
- answer_citations: The response includes proper citations or sources when required
- completeness: The response is complete and covers all key aspects of the request
- tool_call_search: Is the search tool invoked?

Output 1/0 corresponding to true/false for passing each check and provide a short explanation
for your judgment

Then include a final 'verdict' - again 1/0 corresponding to true/false - of whether this eval case
has passed considering the entire checklist.
""".strip()

In [152]:
evals_user_prompt_format = """
<INSTRUCTIONS>{instructions}</INSTRUCTIONS>
<QUESTION>{question}</QUESTION>
<ANSWER>{answer}</ANSWER>
<LOG>{log}</LOG>
""".strip()

In [153]:
tech_assistant_system_prompt = """
    You are a helpful assistant for a course.
    Use the search tool to find relevant information from the course materials
    before answering questions.
    If you can find specific information through search, state that you have gathered some relevant
    information from the search, explicitly include references under "References:" at the top of your answer
    to that information by citing something like the filename, the URL to the github repo or the section name
    for the chunk(s) you used info from and use it to provide accurate answers.
    Always include the references at the top of your answer in the below format.
    FORMAT: 
    References:
    1. [REPO NAME]-[SECTION NAME] -> [FULL GITHUB URL]
    2. [REPO NAME]-[SECTION NAME] -> [FULL GITHUB URL]
    ...
    
    If the search doesn't return relevant results, let the user know that and provide
    general guidance based on your existing knowledge.
"""

### Define our tools
This time let's try to define the search tool using vector search

In [119]:
def load_object(file_path):
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File {file_path} does not exist")
    
    if not file_path.endswith('.pkl'):
        raise ValueError(f"File {file_path} is not a pickle file")
    
    with open(file_path, 'rb') as f:
        loaded_object = pickle.load(f)
    
    return loaded_object

In [120]:
vector_index = load_object("data/section_chunks_vec_index.pkl")

In [121]:
embedding_model = SentenceTransformer('multi-qa-distilbert-cos-v1')

In [122]:
def vector_search(query: str) -> List[Dict]:
    query_emb = embedding_model.encode(query)
    results = vector_index.search(query_emb, num_results=5)
    return results
    # return "\n\n".join([r['content'] for r in results]).strip()

### Define logging functions

In [159]:
def serializer(obj): 
    if isinstance(obj, datetime):
        return obj.isoformat()  
    elif isinstance(obj, Path):
        return str(obj)
    elif isinstance(obj, Callable):
        return obj.__name__
    else:
        raise TypeError(f"Type {type(obj)} not serializable")

In [160]:
def log_to_file(agent, messages, source="user", log_dir=LOG_DIR):
    tool_list = []
    for ts in agent.toolsets:
        tool_list.extend(ts.tools)
    
    time_now = datetime.now()
    random_hex = secrets.token_hex(4)
    filename = f"{agent.name}_{time_now.strftime('%Y%m%d_%H%M%S')}_{random_hex}.json"
    log_file = log_dir / filename
    
    log_entry = {
        'agent': agent.name,
        'source': source,
        'messages': pda.ModelMessagesTypeAdapter.dump_python(messages),
        'model': agent.model.model_name,
        'provider': agent.model.system,
        'instructions': agent.instructions,
        'tools': tool_list,
        'log_file': log_file,
        'created_at': time_now.isoformat()
    }
    
    with log_file.open('w', encoding='utf-8') as f:
        f.write(json.dumps(log_entry, indent=2, default=serializer))
        
    return log_file
        

In [155]:
def load_log(log_file):
    with log_file.open('r', encoding='utf-8') as f:
        log_data = json.load(f)
    
    return log_data

### Now, define our Agents

In [125]:
# Question Generation Agent

quesgen_agent = pda.Agent(
    name = "quesgen",
    instructions = quesgen_system_prompt.format(n_questions=NUM_QUESTIONS),
    model = "gpt-4o-mini",
    output_type = QuestionList
)

In [126]:
# Tech Assistant

techassistant_agent = pda.Agent(
    name = "techassistant",
    instructions = tech_assistant_system_prompt,
    model = "gpt-4o-mini",
    tools = [vector_search]
)

In [128]:
# Evaluation Agent

evals_agent = pda.Agent(
    name = "evals",
    instructions = evals_system_prompt,
    model = "gpt-3.5-turbo",
    output_type = EvalsChecklist
)

### Execute end to end pipeline

In [129]:
tech_kb = load_object("data/tech_knowledge_base.pkl")

In [130]:
random_indices = random.choices(range(len(tech_kb)), k=2*NUM_QUESTIONS)
sampled_knowledge_base = [doc['content'] for idx, doc in enumerate(tech_kb) if idx in random_indices]
quesgen_source = "Snippet-1"
for i, content in enumerate(sampled_knowledge_base):
    quesgen_source += f"\nSnippet-{i+1}:\n{content}\n"
quesgen_source = quesgen_source.strip()

In [131]:
question_list = await quesgen_agent.run(quesgen_source)
question_list = question_list.output.questions
answer_list = []
eval_result_list = []

In [ ]:
# question = question_list[0]
# answer_response = await techassistant_agent.run(question)
# answer = answer_response.output
# print(answer)

In [162]:
for question in question_list:
    answer_response = await techassistant_agent.run(question)
    answer = answer_response.output
    answer_list.append(answer)
    
    log_file = log_to_file(techassistant_agent, 
                           answer_response.new_messages())
    
    evals_user_prompt = evals_user_prompt_format.format(
        instructions=evals_agent.instructions,
        question=question,
        answer=answer,
        log=load_log(log_file)['messages']
    )
    
    eval_response = await evals_agent.run(evals_user_prompt)
    
    formatted_eval = {
        'verdict': int(eval_response.output.verdict),
        'detailed_checks': [{
            'check': check.check_name,
            'reason': check.justification,
            'result': int(check.check_pass)
        } for check in eval_response.output.checks]
    }
    
    eval_result_list.append(formatted_eval)

In [164]:
eval_result_list[0]

{'verdict': 1,
 'detailed_checks': [{'check': 'instructions_follow',
   'reason': "The agent followed the user's instructions by providing a detailed guide on demonstrating foundational knowledge of data analysis using Python with references.",
   'result': 1},
  {'check': 'instructions_avoid',
   'reason': 'The agent did not engage in any actions it was instructed to avoid.',
   'result': 1},
  {'check': 'answer_relevant',
   'reason': "The response provides a detailed guide on how to demonstrate foundational knowledge of data analysis using Python, directly addressing the user's question.",
   'result': 1},
  {'check': 'answer_clear',
   'reason': 'The answer is clear and structured, offering a step-by-step approach to showcasing foundational knowledge in data analysis using Python.',
   'result': 1},
  {'check': 'answer_citations',
   'reason': 'The response includes references at the top of the answer, providing sources for the information shared.',
   'result': 1},
  {'check': 'co

In [166]:
total_questions = len(question_list)
total_correct = len([r for r in eval_result_list if r['verdict'] == 1])
total_incorrect = total_questions - total_correct
accuracy = total_correct / total_questions

print(f"""
Evaluation Results:\n
Total questions & answers: {total_questions}
Total correct: {total_correct}
Total incorrect: {total_incorrect}
Accuracy: {accuracy:.2f}
""")


Evaluation Results:

Total questions & answers: 66
Total correct: 53
Total incorrect: 13
Accuracy: 0.80



In [168]:
pass_instructions_follow = sum([1 for result in eval_result_list if result['detailed_checks'][0]['result'] == 1])
pass_instructions_avoid = sum([1 for result in eval_result_list if result['detailed_checks'][1]['result'] == 1])
pass_answer_relevant = sum([1 for result in eval_result_list if result['detailed_checks'][2]['result'] == 1])
pass_answer_clear = sum([1 for result in eval_result_list if result['detailed_checks'][3]['result'] == 1])
pass_answer_citations = sum([1 for result in eval_result_list if result['detailed_checks'][4]['result'] == 1])
pass_completeness = sum([1 for result in eval_result_list if result['detailed_checks'][5]['result'] == 1])
pass_tool_call_search = sum([1 for result in eval_result_list if result['detailed_checks'][6]['result'] == 1])

print(f"""
Detailed Evaluation Results:\n
Total questions & answers: {total_questions}
Total questions where "instructions_follow" criteria passed: {pass_instructions_follow}
Accuracy: {float(pass_instructions_follow/total_questions)*100:.2f}%
Total questions where "instructions_avoid" criteria passed: {pass_instructions_avoid}
Accuracy: {float(pass_instructions_avoid/total_questions)*100:.2f}%
Total questions where "answer_relevant" criteria passed: {pass_answer_relevant}
Accuracy: {float(pass_answer_relevant/total_questions)*100:.2f}%
Total questions where "answer_clear" criteria passed: {pass_answer_clear}
Accuracy: {float(pass_answer_clear/total_questions)*100:.2f}%
Total questions where "answer_citations" criteria passed: {pass_answer_citations}
Accuracy: {float(pass_answer_citations/total_questions)*100:.2f}%
Total questions where "completeness" criteria passed: {pass_completeness}
Accuracy: {float(pass_completeness/total_questions)*100:.2f}%
Total questions where "tool_call_search" criteria passed: {pass_tool_call_search}
Accuracy: {float(pass_tool_call_search/total_questions)*100:.2f}%
""")


Detailed Evaluation Results:

Total questions & answers: 66
Total questions where "instructions_follow" criteria passed: 53
Accuracy: 80.30%
Total questions where "instructions_avoid" criteria passed: 59
Accuracy: 89.39%
Total questions where "answer_relevant" criteria passed: 66
Accuracy: 100.00%
Total questions where "answer_clear" criteria passed: 66
Accuracy: 100.00%
Total questions where "answer_citations" criteria passed: 59
Accuracy: 89.39%
Total questions where "completeness" criteria passed: 66
Accuracy: 100.00%
Total questions where "tool_call_search" criteria passed: 58
Accuracy: 87.88%

